# [2장 통합 실습] 공개 API로 게시글 두 건 읽기

아래 문제 설명에 해당하는 완성 코드가 들어 있습니다. 실행하고 자신의 풀이와 비교하세요.

## 만들 것

Private LLM 도우미에 사내 게시글을 연결하기 전에 수집 프로그램의 첫 요청을 점검하려고 합니다. 사내 서버 대신 **JSONPlaceholder의 공개 연습용 글 API**를 사용해, 지정한 작성자의 글 두 건을 읽는 프로그램을 완성하세요.

이 공개 서버의 글은 연습용 가짜 내용입니다. 실제 요청을 보내더라도 사내 규정이나 사실을 수집한 것은 아닙니다. 이번 결과물은 요청 주소·상태·실행 모드와 글의 ID·제목을 보여 주는 실행 결과입니다. LLM 답변이나 파일 저장은 만들지 않습니다.

## 제공 자료와 실행 모드

`MODE`의 기본값은 `live`입니다. 공개 서버에 GET 요청을 한 번 보내며, 접속·상태·응답 확인에 실패하면 제공 샘플로 전환해 `sample_fallback`을 표시합니다. `sample`로 바꾸면 처음부터 네트워크 없이 샘플 두 건을 처리합니다. 샘플 ID는 101과 102이며, 제목은 `교육 신청`, `지원 범위`입니다.

공개 API에는 키가 필요하지 않습니다. 환경 변수와 인증 헤더를 연습하는 짧은 로컬 요청이 별도로 제공되어 있습니다. `PRACTICE_API_KEY`가 없을 때 코드가 연습용 값을 넣으며, 실제 계정의 키를 준비할 필요는 없습니다. 이 헤더는 `MockTransport`가 받는 로컬 요청에만 사용합니다. 이 도구는 서버로 연결하는 대신 제공 함수가 메모리 안에서 HTTP 응답을 돌려주게 합니다.

## 이번 요청의 구성

`BASE_URL`은 서버 주소이고 `/posts`는 글 목록을 조회하는 endpoint입니다. `/posts/1`처럼 주소 경로에 ID를 붙이면 특정 글을 가리키지만, 이번에는 목록에서 조건에 맞는 글을 조회합니다.

| 항목 | 이번 요청에서 사용할 값과 의미 |
| --- | --- |
| method | `GET`: 서버의 글을 읽는다. |
| header | `Accept`에 `application/json`을 넣어 원하는 응답 형식을 알린다. |
| query `userId` | 정수 1: 첫 작성자의 글로 범위를 좁힌다. |
| query `_page` | 정수 1: 첫 페이지를 요청한다. |
| query `_limit` | 정수 2: 한 페이지에 두 건을 요청한다. |
| timeout | 5.0초: 연결이나 응답을 무한정 기다리지 않는다. |

query는 `params` 딕셔너리로 전달합니다. HTTPX가 URL에 필요한 형태로 변환하므로 주소 문자열에 직접 이어 붙이지 않아도 됩니다. GET 요청에는 별도 JSON body를 넣지 않습니다. POST·PUT·PATCH·DELETE처럼 생성·변경·삭제에 쓰이는 method는 이 조회 작업에 필요하지 않습니다.

정상 응답의 맨 바깥은 리스트이며, 글 하나는 `id`, `title`, `body` 등을 가진 딕셔너리입니다. API마다 이 구조가 다르므로 `items`라는 필드가 있을 것이라고 가정하면 안 됩니다.

## TODO 1 — 로컬 인증 헤더 만들기

`auth_headers()`에는 인수가 없습니다. 환경 변수 `PRACTICE_API_KEY`를 읽고 앞뒤 공백을 제거한 뒤, `Authorization`을 키로 가진 딕셔너리를 반환하세요. 값은 `Bearer`, 공백 하나, 읽은 키를 이어 붙인 문자열입니다.

환경 변수가 없거나 공백뿐이면 `RuntimeError`를 발생시키세요. `os.getenv(이름, 기본값)`은 설정이 없을 때 사용할 값을 지정할 수 있고, 문자열의 `.strip()`은 앞뒤 공백을 제거합니다. 키 자체는 출력하지 않습니다.

## TODO 2 — 두 건을 요청하고 상태 확인하기

`request_posts(client)`의 `client`는 준비된 `httpx.Client`입니다. 위 표에 맞는 headers와 params를 구성해 `/posts`로 요청하고, 응답 상태를 확인한 뒤 **HTTPX 응답 객체 자체**를 반환하세요. 여기서는 JSON을 리스트로 바꾸지 않습니다.

클라이언트의 `.get()`에 `headers`, `params`, `timeout`을 전달할 수 있습니다. 응답의 `.raise_for_status()`는 실패 상태를 예외로 알려 줍니다. JSON을 해석하기 전에 이 단계를 거쳐야 합니다.

## TODO 3 — JSON과 필요한 구조 확인하기

`read_posts(response)`는 앞 함수가 반환한 응답 객체를 받아 글 목록을 반환합니다. 순서는 다음과 같습니다.

1. `Content-Type`에 `application/json`이 포함되어 있는지 확인하고, 그렇지 않으면 `ValueError`를 발생시킵니다.
2. 응답의 `.json()`으로 본문을 Python 값으로 바꿉니다. JSON 문법 자체가 잘못되면 여기서 예외가 발생합니다.
3. 맨 바깥이 리스트인지 확인합니다. 아니라면 `ValueError`입니다.
4. 각 항목이 딕셔너리이고 필수 키 `id`, `title`, `body`를 모두 갖는지 확인합니다. 하나라도 어기면 `ValueError`입니다.
5. 확인한 리스트를 반환합니다.

자료형은 `isinstance`, 필수 키 포함 여부는 집합의 `.issubset()`으로 확인할 수 있습니다. 예를 들어 필요한 키 집합이 실제 딕셔너리의 키 안에 모두 들어 있는지를 검사합니다. 이번 검사는 구조와 필수 키까지이며, 본문의 사실 여부나 제목 품질을 판정하지 않습니다.

## 실행하기

시작 코드의 세 `raise NotImplementedError(...)`를 구현하고 위에서부터 실행하세요. 마지막 셀은 먼저 로컬 인증 헤더를 확인하고, 이어 선택한 모드로 글을 읽습니다. 로컬 출력의 `True`는 `Authorization` 값의 `Bearer ` 접두사가 확인되었다는 뜻이며 실제 계정 인증 성공을 뜻하지 않습니다.

우선 기본 `live` 결과의 모드를 확인한 뒤, `MODE`만 `sample`로 바꾸어 제공된 두 건도 확인하세요. 코드를 반복해서 고칠 때는 `sample`을 사용하면 외부 서버에 불필요한 요청을 보내지 않고 비교할 수 있습니다.

## 실행 준비

**Windows + VS Code + PowerShell + Python 3.12 + uv**

아래 자료 ZIP을 내려받아 압축을 풀고, `pyproject.toml`이 있는 폴더를 VS Code로 엽니다. Python과 Jupyter 확장을 설치하고 **터미널 → 새 터미널**에서 PowerShell을 선택하세요. uv가 없다면 먼저 `python -m pip install uv`를 실행합니다.

```powershell
uv sync --python 3.12
uv run python --version
uv run python solution.py
```

버전 출력이 `Python 3.12.x`인지 확인합니다. 제공된 `.python-version`과 `pyproject.toml`도 Python 3.12를 지정합니다. 해당 Python이 없으면 uv가 준비합니다. 별도의 가상환경 활성화 명령은 필요하지 않습니다.

노트북으로 풀려면 `data_api_chapter02_solution.ipynb`를 열고 오른쪽 위 **커널 선택**에서 이 폴더의 `.venv`를 선택합니다. `uv sync`를 마쳤다면 노트북의 패키지 설치 셀은 건너뛰고 준비 코드부터 실행하세요.

**선택: Google Colab**

Colab을 사용할 때는 **파일 → 노트북 업로드**에서 `data_api_chapter02_solution.ipynb`를 엽니다. 필요한 데이터는 노트북에 포함되어 있습니다. 첫 패키지 설치 셀부터 실행하고, 이미 불러온 패키지의 버전 변경 안내가 나오면 런타임을 다시 시작합니다.

```python
%pip install -q "httpx==0.28.1"
```

## 패키지 설치

Windows에서 `uv sync`를 마쳤다면 이 설치 셀은 건너뜁니다. Colab에서는 설치 셀부터 실행하세요.

In [ ]:
%pip install -q "httpx==0.28.1"


## 1. 실행 모드와 제공 샘플

In [2]:
import os
import httpx

MODE = "live"  # 외부 요청 없이 연습하려면 "sample"로 바꿉니다.
BASE_URL = "https://jsonplaceholder.typicode.com"
SAMPLE = [
    {"userId": 1, "id": 101, "title": "교육 신청", "body": "교육 포털에서 신청합니다."},
    {"userId": 1, "id": 102, "title": "지원 범위", "body": "지원 항목은 규정에서 확인합니다."},
]
# 인증 연습 전용 값입니다. 실제 계정의 키를 입력할 필요가 없습니다.
os.environ.setdefault("PRACTICE_API_KEY", "practice-only")


def auth_headers():
    # TODO 1: 공백을 제거한 키로 로컬 요청의 인증 헤더를 만듭니다.
    key = os.getenv("PRACTICE_API_KEY", "").strip()
    if not key:
        raise RuntimeError("PRACTICE_API_KEY가 비어 있습니다.")
    return {"Authorization": f"Bearer {key}"}
    # TODO 1 끝


def local_auth(request):
    # 이 함수는 네트워크 대신 메모리 안에서 요청을 받습니다.
    ok = request.headers.get("Authorization", "").startswith("Bearer ")
    return httpx.Response(200 if ok else 401, json={"authorized": ok})




## 2. 공개 API 요청과 응답 검사

In [3]:
def request_posts(client):
    # TODO 2: 첫 작성자의 글 중 첫 페이지 두 건만 요청합니다.
    headers = {"Accept": "application/json"}
    params = {"userId": 1, "_page": 1, "_limit": 2}
    response = client.get("/posts", headers=headers, params=params, timeout=5.0)
    # 404의 JSON 오류 본문을 정상 글 목록처럼 읽지 않도록 HTTP 상태부터 확인합니다.
    response.raise_for_status()
    return response
    # TODO 2 끝


def read_posts(response):
    # TODO 3: JSON 문법과 우리가 필요한 데이터 구조를 따로 확인합니다.
    if "application/json" not in response.headers.get("Content-Type", ""):
        raise ValueError("JSON 응답이 아닙니다.")
    items = response.json()
    if not isinstance(items, list):
        raise ValueError("응답은 글 목록이어야 합니다.")
    # 목록 자체가 맞아도 내부에 숫자나 필드가 빠진 객체가 섞일 수 있습니다.
    for item in items:
        if not isinstance(item, dict) or not {"id", "title", "body"}.issubset(item):
            raise ValueError("글의 필수 필드가 없습니다.")
    return items
    # TODO 3 끝


def sample_response(request):
    return httpx.Response(200, json=SAMPLE)




## 3. 로컬 인증 확인 후 선택한 모드로 읽기

In [4]:
with httpx.Client(base_url="https://practice.invalid",
                  transport=httpx.MockTransport(local_auth)) as local:
    check = local.get("/check", headers=auth_headers())
    check.raise_for_status()
    print("로컬 인증 연습:", check.json()["authorized"])

if MODE not in {"live", "sample"}:
    raise ValueError("MODE는 live 또는 sample이어야 합니다.")
used_mode = MODE
transport = httpx.MockTransport(sample_response) if MODE == "sample" else None
try:
    # 로컬 인증 헤더를 이 공개 API 클라이언트에 전달하지 않습니다.
    with httpx.Client(base_url=BASE_URL, transport=transport) as client:
        response = request_posts(client)
        items = read_posts(response)
except (httpx.HTTPError, ValueError) as error:
    print("요청/응답 확인 실패:", type(error).__name__)
    used_mode = "sample_fallback"
    with httpx.Client(base_url=BASE_URL,
                      transport=httpx.MockTransport(sample_response)) as client:
        response = request_posts(client)
        items = read_posts(response)

print("실행 모드:", used_mode)
print("요청:", response.request.method, response.request.url)
print("응답:", response.status_code, "| 글 수:", len(items))
for item in items:
    print(item["id"], "|", item["title"])


로컬 인증 연습: True
실행 모드: live
요청: GET https://jsonplaceholder.typicode.com/posts?userId=1&_page=1&_limit=2
응답: 200 | 글 수: 2
1 | sunt aut facere repellat provident occaecati excepturi optio reprehenderit
2 | qui est esse


## 결과 확인

완성 후 요청 method는 `GET`, 경로는 `/posts`여야 합니다. 출력된 query에서 `userId=1`, `_page=1`, `_limit=2`를 확인하세요. 공개 요청에 `Authorization`을 추가하지 않았는지도 코드에서 확인합니다.

`sample`에서는 상태 200, 글 수 2, `101 | 교육 신청`, `102 | 지원 범위`가 나와야 합니다. `live`에서는 공개 서버의 글이 나오므로 제목을 샘플 제목과 비교하지 않습니다. 실패 후 샘플로 전환했다면 `sample_fallback`이라고 표시되어야 합니다.

로컬 인증의 빈 값도 확인해 보세요. 첫 셀을 실행한 다음 `PRACTICE_API_KEY`에 공백 문자열을 넣고 `auth_headers()`만 호출하면 `RuntimeError`가 나야 합니다. 확인 후 연습용 문자열로 되돌립니다. 환경 변수 값을 출력하거나 공개 요청에 보낼 필요는 없습니다.

## 짧은 관찰 메모

1. 이번 실행은 공개 서버 응답과 제공 샘플 중 어느 것을 사용했나요? 결과에서 그 근거를 찾고, 샘플 성공만으로 공개 서버 연결까지 확인했다고 할 수 없는 이유를 적으세요.
2. 상태가 404인 JSON 응답과, 상태는 200이지만 본문이 객체 하나인 응답은 각각 어디서 멈춰야 하나요?
3. `params`와 `headers`는 각각 무엇을 지정했나요? 인증에 쓰는 환경 변수를 읽었는데도 공개 서버에 그 값이 전달되지 않는 이유를 설명하세요.

이 장은 약 45분 동안 TODO 구현과 결과 확인, 관찰 메모 작성까지 진행합니다. 완성한 코드 또는 노트북에 실행 모드, 요청 경로·query, 상태 코드와 글 수가 보이는 결과를 붙이세요. 로컬 인증의 공백 값 검사 결과와 404·잘못된 목록 구조가 각각 멈추는 위치를 관찰 메모에 남깁니다. 인증 값은 기록하지 않습니다.

## 해설

## 요청과 해석을 나눈 이유

`request_posts`는 요청을 보내 상태까지 확인하고 응답 객체를 반환합니다. `read_posts`가 그 객체에서 본문을 읽습니다. 두 함수가 모두 리스트를 반환한다고 생각하면 뒤에서 `.headers`나 `.json()`을 사용할 때 오류가 납니다. 입력과 반환값의 종류를 먼저 구분하세요.

GET은 조회를 위한 method이고 `userId`, `_page`, `_limit`는 조회 범위입니다. `Accept`는 어떤 글을 고를지 정하지 않고 어떤 응답 형식을 원하는지 알립니다. 헤더와 query는 서로 바꿔 넣을 수 없습니다. Requests도 유사한 방식으로 요청할 수 있지만, 이 코드에서는 HTTPX Client가 서버 주소와 연결을 관리합니다.

## 상태 → JSON → 구조 순서

오류 응답도 JSON일 수 있습니다. 상태가 404인데 본문만 읽으면 오류 안내 객체를 정상 글처럼 취급할 수 있으므로 먼저 `.raise_for_status()`를 호출합니다. 401은 인증, 403은 접근 허용, 429는 요청 제한과 관련된 상태입니다. 400은 보낸 요청을, 500대 상태는 서버 쪽 처리를 살펴볼 단서가 됩니다. 이 장에서는 자동 재시도 없이 실패 모드를 표시하고 제공 샘플을 사용합니다.

상태 200은 요청 처리 결과이지 데이터 구조 보증은 아닙니다. Content-Type이 JSON인지, JSON 문법을 읽을 수 있는지, 읽은 값이 글 목록인지 차례로 확인합니다. `.issubset()`은 필요한 키가 모두 있는지 검사하므로 `userId` 같은 추가 키가 있어도 허용합니다. 딕셔너리인지 먼저 확인하면 글 대신 문자열이나 숫자가 들어온 경우에도 구조 오류로 다룰 수 있습니다.

이 검사를 통과했다고 데이터가 사내 질문의 근거로 적합하다는 뜻은 아닙니다. 빈 제목, 내용의 사실 여부, 중복 여부 등은 별도 판단이 필요합니다. 이 공개 API의 내용은 처음부터 연습용 가짜 글입니다.

## 환경 변수와 로컬 인증 요청

`os.getenv`의 기본값을 빈 문자열로 정하면 변수가 없을 때도 `.strip()`을 사용할 수 있습니다. 공백만 있던 값은 빈 문자열이 되고, 이때 요청 전에 `RuntimeError`를 발생시킵니다. 시작 설정의 `setdefault`는 변수가 없을 때만 연습용 값을 넣으므로 이미 공백이 들어 있다면 덮어쓰지 않습니다.

완성한 헤더는 로컬 `MockTransport`에만 전달합니다. 제공 함수는 Bearer 형태의 헤더가 있는지 확인해 응답합니다. 실제 사용자나 키의 권한을 검증하는 서버는 아닙니다. 공개 클라이언트는 별도로 만들고 `Accept`만 전달하므로, 로컬 인증 연습이 외부에 키를 보내는 동작으로 이어지지 않습니다.

환경 변수가 비어 발생하는 `RuntimeError`는 설정 문제입니다. 실제 수집의 실패를 샘플로 바꾸는 코드보다 앞에서 멈춥니다. 반면 접속·timeout·HTTP 상태 오류 또는 응답의 JSON·구조 오류는 수집 부분에서 처리합니다. 출력에는 오류의 종류를 남기고 키나 인증 헤더 원문은 남기지 않습니다.

## 관찰 메모 예시

실제 요청이 성공했다면 “출력 모드가 live이고 공개 서버가 반환한 ID와 제목이 보인다”고 적을 수 있습니다. `sample_fallback`이면 “실제 요청 또는 응답 확인에 실패했고 로컬 샘플 두 건을 처리했다”고 적습니다. 샘플에 붙어 있는 요청 URL은 HTTPX가 만든 요청 정보이며, 그 주소로 네트워크 연결을 했다는 증거는 아닙니다.

404 JSON 응답은 TODO 2의 상태 검사에서 멈춥니다. 200이지만 객체 하나인 JSON 응답은 상태와 JSON 문법 검사를 통과한 뒤 TODO 3의 리스트 검사에서 멈춥니다.

마지막 메모는 “params는 작성자와 페이지 범위를 정하고, headers는 원하는 형식을 알린다. 인증 헤더는 로컬 클라이언트에만 전달하고 공개 클라이언트에는 전달하지 않는다”는 내용이면 됩니다.